In [3]:
import pandas as pd

In [5]:
df = pd.read_csv('1hour_jan_aug.csv')
df.head()

,time,open,high,low,close,EMA,EMA.1,Volume
0,1737417600,2710.305,2711.705,2702.815,2705.665,2707.327258,2707.943741,17803
1,1737421200,2705.710,2719.408,2703.215,2718.010,2708.298416,2709.956993,34489
2,1737424800,2717.950,2725.775,2717.810,2724.835,2709.801742,2712.932594,25563
3,1737428400,2724.845,2727.015,2721.295,2724.250,2711.115220,2715.196075,18958
4,1737432000,2724.265,2727.925,2723.075,2727.395,2712.595200,2717.635860,13404


In [ ]:
df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)

df["time_bd"] = df["time"].dt.tz_convert("Asia/Dhaka")

tokyo_df = df[
    (df["time_bd"].dt.hour >= 6) &
    (df["time_bd"].dt.hour < 8)
].copy()

tokyo_df["movement"] = (tokyo_df["close"] - tokyo_df["open"]) / 0.01



month_input = int(input("Enter month number (1 for Jan, 2 for Feb, etc.): "))

filtered_df = tokyo_df[tokyo_df["time_bd"].dt.month == month_input]

filtered_df.tail(10)

In [7]:
import yfinance as yf
import pandas as pd

df = yf.download(
    tickers="GC=F",
    interval="1d",
    start="2024-01-01",
    end="2025-12-01",
    progress=False
)


df.reset_index(inplace=True)

# flatten multi-index columns
df.columns = df.columns.get_level_values(0)

# ensure datetime
df["Date"] = pd.to_datetime(df["Date"])

# add day name
df["day_name"] = df["Date"].dt.day_name()

# add candle direction
df["candle_body"] = (df["Close"] - df["Open"]) / 0.01

# add price movement
df["price_movement_pips"] = (df["High"] - df["Low"]) / 0.01

# group by day_name and sum absolute movement
daily_total = df.groupby("day_name")["price_movement_pips"].apply(
    lambda x: x.abs().sum())
# sort descending to get most active weekdays first
daily_total = daily_total.sort_values(ascending=False)

print("Monthly Gold Report (Total Daily Movement in Pips)")
print(daily_total)


report = daily_total.reset_index()
report.columns = ["Weekday", "Total_Movement_Pips"]
report["Rank"] = report["Total_Movement_Pips"].rank(
    ascending=False, method="dense").astype(int)
report = report.sort_values("Rank")
print(report)

Monthly Gold Report (Total Daily Movement in Pips)
day_name
Friday       353620.031738
Thursday     338800.183105
Monday       324940.026855
Tuesday      312399.987793
Wednesday    302219.885254
Name: price_movement_pips, dtype: float64
     Weekday  Total_Movement_Pips  Rank
0     Friday        353620.031738     1
1   Thursday        338800.183105     2
2     Monday        324940.026855     3
3    Tuesday        312399.987793     4
4  Wednesday        302219.885254     5
